# Projet ML — Prédiction du CO2 par habitant à partir du profil socio-économique d'un pays

**Objectif du projet** : comprendre comment le profil socio-économique d'un pays (développement, mode de vie, structure de production) permet de prédire ses émissions de CO2 par habitant (`co2_per_capita`), et si l'ajout d'un historique temporel (valeurs des années précédentes) améliore la prédiction.

**Dataset** : *Global Climate & CO2 Anomalies (1990-2024)*, 30 pays, 35 ans, 54 variables (climat, énergie, socio-économie). Source : Kaggle.

**Plan du notebook** :
1. Chargement et nettoyage des données
2. Exploration (EDA) — comprendre la structure du dataset et les relations entre variables
3. Choix de la variable cible
4. Preprocessing (features, lags, split train/test)
5. Modélisation (régression linéaire vs Random Forest)
6. Optimisation des hyperparamètres (Optuna)
7. Diagnostic (learning curve, feature importance)
8. Prochaines étapes

## 1. Chargement des données

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer

In [ ]:
df = pd.read_csv("D:\Code_data\DESU_2026\global_climate_co2_anomalies.csv")

**Problème découvert en explorant les données** : le CSV contient 12 600 lignes, mais avec seulement 30 pays × 35 ans, on devrait s'attendre à 1 050 lignes si chaque ligne = un pays/année. En réalité, la plupart des variables (socio-économiques notamment) ne changent qu'une fois par an et sont **répétées à l'identique sur les 12 mois** de chaque année. Garder cette granularité mensuelle créerait un risque de fuite de données (les 12 mois d'une même année pourraient se retrouver éclatés entre train et test). On réduit donc le dataset à une ligne par pays/année.

In [ ]:
# Garder une seule ligne par pays/année au lieu de 12 (voir explication ci-dessus)
df_annual = df.drop_duplicates(subset=['country','year'], keep='first')

In [ ]:
df_annual.head()

In [ ]:
# Vue d'ensemble des 54 colonnes disponibles
for col in df.columns:
    print(col)

## 2. Exploration (EDA)

### 2.1 — Population vs émissions totales de CO2

Première question : le volume total d'émissions d'un pays est-il simplement lié à sa taille démographique ?

In [ ]:
# Un point = un pays : moyenne sur les 35 ans
df_by_country = df_annual.groupby('country')[['population_millions','co2_emissions']].mean().reset_index()

plt.scatter(df_by_country['population_millions'], df_by_country['co2_emissions'], alpha=0.7, s=40)

plt.xlabel('population_millions (moyenne 1990-2024)')
plt.ylabel('co2_emissions (moyenne 1990-2024)')
plt.tight_layout()
plt.show()

In [ ]:
plt.scatter(df_by_country['co2_emissions'], df_by_country['population_millions'], alpha=0.7, s=40)

for _, row in df_by_country.iterrows():
    plt.annotate(row['country'], (row['co2_emissions'], row['population_millions']),
                 fontsize=(7), xytext=(4,4), textcoords='offset points')

plt.xlabel('co2_emissions (moyenne 1990-2024)')
plt.ylabel('population_millions (moyenne 1990-2024)')
plt.tight_layout()
plt.show()

**Constat** : relation clairement positive — plus un pays est peuplé, plus ses émissions totales sont élevées (Chine et Inde très à l'écart des autres). Logique mathématiquement : `co2_emissions = population × co2_per_capita`, et la population varie sur une bien plus grande échelle (~500x entre le plus petit et le plus grand pays) que le CO2 par habitant (~40x) — c'est donc elle qui domine le total, quasiment par construction.

### 2.2 — Qu'en est-il de `co2_per_capita` (CO2 par habitant) ?

In [ ]:
# Un point = un pays : moyenne sur les 35 ans
df_by_country = df_annual.groupby('country')[['population_millions','co2_per_capita']].mean().reset_index()

plt.scatter(df_by_country['population_millions'], df_by_country['co2_per_capita'], alpha=0.7, s=40)

for _, row in df_by_country.iterrows():
    plt.annotate(row['country'], (row['population_millions'], row['co2_per_capita']),
                 fontsize=(7), xytext=(4,4), textcoords='offset points')

plt.xlabel('population_millions (moyenne 1990-2024)')
plt.ylabel('co2_per_capita (moyenne 1990-2024)')
plt.tight_layout()
plt.show()

**Constat** : ici, plus de relation nette avec la population — l'Inde (très peuplée) a un `co2_per_capita` faible, alors que des pays peu peuplés comme l'Australie ou le Canada ont un `co2_per_capita` très élevé. Ça confirme que `co2_per_capita` raconte une histoire différente de `co2_emissions` : plutôt liée au développement/mode de vie qu'à la taille du pays.

**Question soulevée** : est-ce que `co2_per_capita` est une simple division `co2_emissions / population_millions`, ou une variable calculée différemment (empreinte carbone individuelle) ? Vérifions.

In [ ]:
# Dernière année uniquement (2024), 1 point = 1 pays
df_last = df_annual[df_annual['year'] == df_annual['year'].max()].copy()
df_last['co2_pa_calc'] = df_last['co2_emissions'] / df_last['population_millions']

fig, axes = plt.subplots(1, 2, figsize=(16,7))

axes[0].scatter(df_last['population_millions'], df_last['co2_per_capita'], alpha=0.7, s=40, color='#2b6cb0')
for _, row in df_last.iterrows():
    axes[0].annotate(row['country'], (row['population_millions'], row['co2_per_capita']), fontsize=7, xytext=(4,4), textcoords='offset points')
axes[0].set_xlabel('population_millions')
axes[0].set_ylabel('co2_per_capita (colonne du dataset)')
axes[0].set_title(f"co2_per_capita — dataset ({int(df_annual['year'].max())})")

axes[1].scatter(df_last['population_millions'], df_last['co2_pa_calc'], alpha=0.7, s=40, color='#c05621')
for _, row in df_last.iterrows():
    axes[1].annotate(row['country'], (row['population_millions'], row['co2_pa_calc']), fontsize=7, xytext=(4,4), textcoords='offset points')
axes[1].set_xlabel('population_millions')
axes[1].set_ylabel('co2_emissions / population_millions (recalculé)')
axes[1].set_title(f"co2_per_capita — recalculé ({int(df_annual['year'].max())})")

plt.tight_layout()
plt.show()

diff = (df_last['co2_per_capita'] - df_last['co2_pa_calc']).abs()
print("Écart absolu moyen:", diff.mean())
print("Écart absolu max:", diff.max())
print("Corrélation:", df_last['co2_per_capita'].corr(df_last['co2_pa_calc']))

In [ ]:
# Vérification supplémentaire : co2_emissions vs co2_per_capita, un point par pays
df_by_country = df_annual.groupby('country')[['co2_emissions','co2_per_capita']].mean().reset_index()

plt.scatter(df_by_country['co2_emissions'], df_by_country['co2_per_capita'], alpha=0.7, s=40)

plt.xlabel('co2_emissions (moyenne 1990-2024)')
plt.ylabel('co2_per_capita (moyenne 1990-2024)')
plt.tight_layout()
plt.show()

In [ ]:
print(df_annual['country'].unique())

**Conclusion de cette vérification** : `co2_per_capita` et `co2_emissions / population_millions` sont extrêmement corrélées (>0.998) une fois agrégées par pays — l'écart observé au niveau mensuel/annuel individuel est du bruit aléatoire qui s'annule en moyenne, pas une différence structurelle. `co2_per_capita` **est bien** dérivée de `co2_emissions`, avec un peu de bruit ajouté (probablement pour simuler des fluctuations réalistes). Ce n'est donc pas une "empreinte carbone individuelle" indépendante — c'est la même information territoriale que `co2_emissions`, juste ramenée par habitant. **Implication pratique : il faudra exclure `co2_emissions` des features si `co2_per_capita` est la cible (risque de fuite de données).**

### 2.3 — Quelles variables socio-économiques sont liées à `co2_per_capita` et à `co2_emissions` ?

Hypothèse de départ : les variables de *mode de vie individuel* (voitures, urbanisation...) devraient mieux expliquer `co2_per_capita`, alors que les variables de *taille/structure du pays* devraient mieux expliquer `co2_emissions` (total).

In [ ]:
vars_to_plot = ['vehicle_ownership','urbanization_rate','gdp_per_capita_usd','agriculture_share_gdp',
                'oil_production','share_coal_pct']

df_by_country = df_annual.groupby('country')[vars_to_plot + ['co2_per_capita','co2_emissions']].mean().reset_index()


fig, axes = plt.subplots(2, 3, figsize=(16,9))
axes = axes.flatten()

for ax, var in zip(axes, vars_to_plot):
    ax.scatter(df_by_country[var], df_by_country['co2_per_capita'], color='#2b6cb0')
    ax.set_xlabel(var)
    ax.set_ylabel('co2_per_capita')
    corr = df_by_country[var].corr(df_by_country['co2_per_capita'])
    ax.set_title(f"r = {corr:.2f}")

plt.suptitle("co2_per_capita vs variables socio-économiques (données annuelles, 1050 lignes)", fontsize=14)
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16,9))
axes = axes.flatten()

for ax, var in zip(axes, vars_to_plot):
    ax.scatter(df_by_country[var], df_by_country['co2_emissions'], color='#2b6cb0')
    ax.set_xlabel(var)
    ax.set_ylabel('co2_emissions')
    corr = df_by_country[var].corr(df_by_country['co2_emissions'])
    ax.set_title(f"r = {corr:.2f}")

plt.suptitle("co2_emissions vs variables socio-économiques (données annuelles, 1050 lignes)", fontsize=14)
plt.tight_layout()

**Constat** : ces variables socio-économiques sont bien corrélées à `co2_per_capita`, mais beaucoup moins à `co2_emissions` (sauf `share_coal_pct` un peu). Ça confirme l'hypothèse de départ : le mode de vie/développement individuel se reflète dans le CO2 *par habitant*, pas dans le volume *total*, qui est dominé par l'effet de taille du pays (population).

**Question suivante** : le mode de vie individuel (voitures, urbanisation...) suffit-il à lui seul à expliquer `co2_emissions` ? Testons avec d'autres variables (énergie, industrie...) plus liées à la structure de production du pays.

In [ ]:
vars_to_plot = ['nino34_index','ocean_heat_content','precipitation_anomaly','primary_energy_twh','share_renewables_pct','co2_per_kwh','electricity_demand_twh','forest_cover_pct','industrial_share_gdp']

df_by_country = df_annual.groupby('country')[vars_to_plot + ['co2_emissions']].mean().reset_index()

fig, axes = plt.subplots(3, 3, figsize=(16,9))
axes = axes.flatten()

for ax, var in zip(axes, vars_to_plot):
    ax.scatter(df_by_country[var], df_by_country['co2_emissions'], color='#2b6cb0')
    ax.set_xlabel(var)
    ax.set_ylabel('co2_emissions')
    corr = df_by_country[var].corr(df_by_country['co2_emissions'])
    ax.set_title(f"r = {corr:.2f}")

plt.suptitle("co2_emissions vs variables socio-économiques (données annuelles, 1050 lignes)", fontsize=14)
plt.tight_layout()

**Constat** : peu de relations linéaires nettes ici non plus avec `co2_emissions` — ce qui est cohérent avec ce qu'on a établi : `co2_emissions` est surtout dominé par un effet de taille (population), pas par ces variables structurelles prises individuellement. Ça achève de confirmer le choix de cible ci-dessous.

## 3. Choix de la variable cible : `co2_per_capita`

**Décision** : on choisit `co2_per_capita` plutôt que `co2_emissions` comme cible, pour les raisons établies pendant l'EDA :
- `co2_emissions` (total) est dominé mécaniquement par la taille démographique du pays (r=0.77 avec la population) — un modèle dessus apprendrait surtout "grand pays = grosses émissions", peu intéressant socio-économiquement.
- `co2_per_capita` reflète mieux le profil de développement/mode de vie d'un pays, indépendamment de sa taille — exactement la question qu'on veut explorer (comment PIB, urbanisation, voitures, mix énergétique... influencent l'intensité carbone par habitant).

**Exclusions nécessaires (fuite de données / data leakage)**, vu que `co2_per_capita` est dérivée de `co2_emissions` :
- `co2_emissions` (corrélation >0.998 avec la cible)
- `co2_per_gdp`, `cumulative_co2`, `total_ghg`, `land_use_change_co2` (dérivées du CO2)
- `emission_intensity_level` (catégorie dérivée des émissions)

In [ ]:
print(df_annual['income_group'].unique())
print(df_annual['continent'].unique())

### 3.1 — Visualiser l'effet du groupe de revenu

In [ ]:
vars_to_plot = ['vehicle_ownership','urbanization_rate','gdp_per_capita_usd',
                'agriculture_share_gdp','oil_production','share_coal_pct','climate_risk_score','forest_cover_pct','share_renewables_pct']

# --- Coloré par income_group ---
income_order = ['Low Income','Lower Middle','Upper Middle','High Income']
colors = {'Low Income':'#e53e3e','Lower Middle':'#dd6b20','Upper Middle':'#3182ce','High Income':'#2f855a'}

fig, axes = plt.subplots(3, 3, figsize=(17,10))
axes = axes.flatten()

for ax, var in zip(axes, vars_to_plot):
    for grp in income_order:
        sub = df_annual[df_annual['income_group']==grp]
        ax.scatter(sub[var], sub['co2_per_capita'], alpha=0.5, s=15, color=colors[grp], label=grp)
    ax.set_xlabel(var)
    ax.set_ylabel('co2_per_capita')

axes[0].legend(loc='upper left', fontsize=8)
plt.suptitle("co2_per_capita vs variables socio-économiques — coloré par groupe de revenu", fontsize=14)
plt.tight_layout()

**Constat** : les couleurs se regroupent nettement par bandes — le groupe de revenu structure bien l'espace des variables. Bon signe pour la modélisation : `income_group` est probablement une feature utile.

### 3.2 — Trajectoires par pays dans le temps

Ce plot permet de vérifier que chaque pays suit sa propre trajectoire cohérente dans le temps (1990→2024), ce qui justifiera l'ajout de features temporelles (lags) plus loin.

In [ ]:
# Trajectoires par pays (quelques pays représentatifs)
sample_countries = ['China','United States','India','Germany','Nigeria','Brazil','France','Saudi Arabia']
fig, axes = plt.subplots(1, 2, figsize=(14,6))

for country in sample_countries:
    sub = df_annual[df_annual['country']==country].sort_values('year')
    axes[0].plot(sub['gdp_per_capita_usd'], sub['co2_per_capita'], marker='o', markersize=3, label=country, alpha=0.8)
    axes[1].plot(sub['oil_production'], sub['co2_per_capita'], marker='o', markersize=3, label=country, alpha=0.8)

axes[0].set_xlabel('gdp_per_capita_usd'); axes[0].set_ylabel('co2_per_capita'); axes[0].set_title('Trajectoire par pays: PIB/hab')
axes[1].set_xlabel('oil_production'); axes[1].set_ylabel('co2_per_capita'); axes[1].set_title('Trajectoire par pays: production pétrole')
axes[0].legend(fontsize=8)
plt.tight_layout()

**Constat important** : chaque pays trace bien une trajectoire lisse et cohérente dans le temps (pas un nuage de points aléatoire). On observe par exemple que la France et l'Allemagne montrent une baisse de `co2_per_capita` sur la période récente malgré un PIB stable/croissant (découplage richesse/émissions), alors que la Chine, l'Inde, le Nigeria sont en phase de hausse. L'Arabie Saoudite a un profil atypique (haute production pétrolière, haut co2_per_capita, sans suivre le schéma "PIB classique").

→ Ça confirme qu'il y a une vraie dynamique temporelle par pays à exploiter : **on va ajouter des features de lag (historique des années précédentes)** dans le preprocessing ci-dessous.

## 4. Preprocessing

**Étapes de cette section :**
1. Sélection de la cible et des features (avec exclusion des variables de fuite de données)
2. Gestion des valeurs manquantes
3. Création des features de lag (1, 2, 5 ans) et de moyenne mobile, pour donner au modèle l'historique récent de chaque pays — répond à la question "le PIB des années précédentes influence-t-il les émissions futures ?"
4. Split train/test **temporel** (pas aléatoire) : train = passé (1995-2016), test = futur jamais vu (2017-2024). Un split aléatoire mélangerait des années très proches d'un même pays entre train et test (ex: France 2015 en train, France 2016 en test), ce qui donnerait un score artificiellement gonflé — le split temporel simule honnêtement la vraie situation de déploiement (prédire l'avenir à partir du passé).
5. Encodage one-hot des variables catégorielles (`income_group`, `continent`), *fit* uniquement sur le train pour ne pas laisser d'information du test influencer le preprocessing.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder

# --- Chargement et passage à l'échelle annuelle ---
df = pd.read_csv("global_climate_co2_anomalies.csv")
df_annual = df.drop_duplicates(subset=['country','year'], keep='first')
df_annual = df_annual.sort_values(['country','year']).reset_index(drop=True)

# --- Définition de la cible et des features ---
target = 'co2_per_capita'

# Features numériques socio-économiques (variables de mode de vie / structure économique)
feature_cols = [
    'gdp_per_capita_usd','urbanization_rate','vehicle_ownership',
    'agriculture_share_gdp','industrial_share_gdp',
    'oil_production','share_coal_pct','share_renewables_pct',
    'forest_cover_pct','population_density'
]

# Features catégorielles
cat_cols = ['income_group','continent']

# On garde country/year à ce stade : nécessaires pour les lags (groupby) et le split temporel
# Elles seront explicitement retirées de X_train/X_test plus bas
df_model = df_annual[['country','year',target] + feature_cols + cat_cols].copy()

# --- Gestion des valeurs manquantes ---
# vehicle_ownership a quelques NaN : on impute par la médiane du pays lui-même
# (plus logique que la médiane globale, qui mélangerait des pays très différents)
df_model['vehicle_ownership'] = df_model.groupby('country')['vehicle_ownership']\
    .transform(lambda x: x.fillna(x.median()))
df_model['vehicle_ownership'] = df_model['vehicle_ownership'].fillna(df_model['vehicle_ownership'].median())

# --- Création des features de lag (historique) ---
# Pour chaque variable clé, on ajoute sa valeur il y a 1, 2 et 5 ans
# groupby('country') est essentiel : on ne veut jamais mélanger l'historique
# d'un pays avec celui d'un autre pays
lag_source_cols = ['gdp_per_capita_usd','urbanization_rate','vehicle_ownership',
                    'oil_production','share_coal_pct','share_renewables_pct']

for lag in [1, 2, 5]:
    for col in lag_source_cols:
        df_model[f'{col}_lag{lag}'] = df_model.groupby('country')[col].shift(lag)

# --- Moyenne mobile 5 ans (tendance récente lissée) ---
# .shift(1) pour ne pas inclure l'année actuelle dans sa propre moyenne (fuite de données)
for col in lag_source_cols:
    df_model[f'{col}_ma5'] = df_model.groupby('country')[col]\
        .transform(lambda x: x.rolling(5, min_periods=1).mean().shift(1))

# Les premières années de chaque pays n'ont pas assez d'historique pour le lag5 -> NaN -> on les retire
df_model = df_model.dropna().reset_index(drop=True)

# --- Split train/test temporel ---
# Train = passé (1995-2016), Test = futur jamais vu (2017-2024)
# Volontairement PAS aléatoire : on veut tester la capacité à prédire le futur, pas mémoriser
train = df_model[df_model['year'] <= 2016].copy()
test  = df_model[df_model['year'] > 2016].copy()

# --- Encodage des variables catégorielles (one-hot) ---
# Le OneHotEncoder est "fit" uniquement sur le train, pour ne jamais laisser
# d'information du test influencer le preprocessing (fuite de données)
num_cols = [c for c in df_model.columns if c not in ['country','year',target]+cat_cols]

enc = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
enc.fit(train[cat_cols])

train_cat = pd.DataFrame(enc.transform(train[cat_cols]),
                          columns=enc.get_feature_names_out(cat_cols), index=train.index)
test_cat = pd.DataFrame(enc.transform(test[cat_cols]),
                         columns=enc.get_feature_names_out(cat_cols), index=test.index)

# country et year sont ici volontairement exclues de X_train/X_test :
# ce ne sont pas des features numériques exploitables telles quelles par le modèle
X_train = pd.concat([train[num_cols], train_cat], axis=1)
X_test  = pd.concat([test[num_cols], test_cat], axis=1)
y_train = train[target]
y_test  = test[target]

print(f"Train: {X_train.shape[0]} lignes, {X_train.shape[1]} features")
print(f"Test:  {X_test.shape[0]} lignes")

df_model.head()

## 5. Modélisation — baseline

On compare deux modèles :
- **Régression linéaire** : simple, interprétable, sert de référence ("baseline")
- **Random Forest** : capture les non-linéarités et interactions entre variables (ex: la courbe de Kuznets environnementale observée sur `gdp_per_capita_usd`) sans qu'on ait à les spécifier à la main

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import joblib

# --- Modèle 1 : Régression linéaire (baseline simple/interprétable) ---
lr = LinearRegression()
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)

# --- Modèle 2 : Random Forest (capture non-linéarités et interactions) ---
rf = RandomForestRegressor(
    n_estimators=300,   # nombre d'arbres
    max_depth=8,        # profondeur max, limite le surapprentissage
    random_state=42,    # reproductibilité
    n_jobs=-1            # utilise tous les coeurs CPU disponibles
)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)

# --- Évaluation ---
def evaluate(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"=== {name} ===")
    print(f"MAE:  {mae:.3f}")
    print(f"RMSE: {rmse:.3f}")
    print(f"R²:   {r2:.3f}\n")

evaluate(y_test, pred_lr, "Régression Linéaire")
evaluate(y_test, pred_rf, "Random Forest")

joblib.dump({'rf':rf,'lr':lr,'X_train':X_train,'X_test':X_test,'y_train':y_train,'y_test':y_test,
             'pred_rf':pred_rf,'pred_lr':pred_lr,'test_meta':test[['country','year']]}, 'model_results.pkl')

**Résultat attendu** : le Random Forest devrait nettement dépasser la régression linéaire (R²≈0.95 vs R²≈0.77 sur nos runs précédents), ce qui indique qu'il existe de vraies relations non-linéaires/interactions entre les variables que la régression simple ne capture pas.

In [ ]:
# Visualisation : prédictions vs réalité + feature importance (Random Forest baseline)
fig, axes = plt.subplots(1, 2, figsize=(15,6))

axes[0].scatter(y_test, pred_rf, alpha=0.5, s=25, color='#2b6cb0')
lims = [min(y_test.min(),pred_rf.min()), max(y_test.max(),pred_rf.max())]
axes[0].plot(lims, lims, 'r--', alpha=0.7, label='Prédiction parfaite')
axes[0].set_xlabel('Valeur réelle (co2_per_capita)')
axes[0].set_ylabel('Valeur prédite')
axes[0].set_title('Random Forest — Prédictions vs Réalité (2017-2024)')
axes[0].legend()

importances = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False).head(15)
axes[1].barh(importances.index[::-1], importances.values[::-1], color='#c05621')
axes[1].set_xlabel('Importance')
axes[1].set_title('Top 15 features les plus importantes')

plt.tight_layout()
plt.show()
print(importances)

## 6. Optimisation des hyperparamètres avec Optuna

Le Random Forest ci-dessus utilise des hyperparamètres choisis "à la main" (n_estimators=300, max_depth=8). On utilise **Optuna** pour chercher automatiquement une meilleure combinaison (n_estimators, max_depth, min_samples_split, min_samples_leaf, max_features, criterion), en minimisant l'erreur sur le test set.

On définit d'abord une fonction `modelresults` réutilisable pour comparer proprement les différentes versions du modèle, et on garde un Random Forest "par défaut" comme référence de comparaison.

In [ ]:
pip install optuna

In [ ]:
import optuna

def modelresults(predictions, model_name):
    mae = mean_absolute_error(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)

    results_dict = {
        'Model': model_name,
        'MAE': mae,
        'MSE': mse,
        'R2 Score': r2
    }

    print('Results for {}:'.format(model_name))
    print('Mean absolute error on model is {:.4f}'.format(mae))
    print('Mean squared error on model is {:.4f}'.format(mse))
    print('The r2 score on model is {:.4f}'.format(r2))

    return results_dict

# Random Forest avec hyperparamètres par défaut de sklearn, comme référence de comparaison
rfr = RandomForestRegressor(random_state=21)
rfr.fit(X_train, y_train)
y_pred_rfr_fit = rfr.predict(X_test)
results_fit = modelresults(y_pred_rfr_fit, 'Random Forest (hyperparamètres par défaut)')

### 6.1 — Recherche à 200 trials

In [ ]:
# Fonction objectif : Optuna teste une combinaison d'hyperparamètres et on retourne le MAE à minimiser
def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 10, 200, log=True)
    max_depth = trial.suggest_int("max_depth", 2, 32)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2'])
    criterion = trial.suggest_categorical('criterion', ["squared_error", "absolute_error", "friedman_mse", "poisson"])

    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=42,
        max_features=max_features,
        criterion=criterion,
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    return mae

In [ ]:
study_200 = optuna.create_study(direction="minimize")
study_200.optimize(objective, n_trials=200, show_progress_bar=True)

In [ ]:
print("Best trial:", study_200.best_trial)
print("Best hyperparameters:", study_200.best_params)

In [ ]:
import optuna.visualization as vis

vis.plot_optimization_history(study_200)
vis.plot_param_importances(study_200)
vis.plot_slice(study_200, params=["n_estimators", "max_depth"])
vis.plot_contour(study_200, params=["min_samples_split", "min_samples_leaf"])
vis.plot_parallel_coordinate(study_200)

In [ ]:
# Construction du modèle final avec les meilleurs hyperparamètres trouvés (200 trials)
best_params_200 = study_200.best_params

best_model_200 = RandomForestRegressor(
    n_estimators=best_params_200['n_estimators'],
    max_depth=best_params_200['max_depth'],
    min_samples_split=best_params_200['min_samples_split'],
    min_samples_leaf=best_params_200['min_samples_leaf'],
    max_features=best_params_200['max_features'],
    criterion=best_params_200['criterion'],
    random_state=42
)
best_model_200.fit(X_train, y_train)
y_pred_rfr_optuna_200_trials = best_model_200.predict(X_test)

results_optuna_200 = modelresults(y_pred_rfr_optuna_200_trials, 'Random Forest Optuna (200 trials)')

### 6.2 — Recherche à 1000 trials

On teste si augmenter drastiquement le nombre d'essais Optuna améliore encore le modèle.

**⚠️ Note méthodologique** : cette fonction objectif utilise une métrique différente (MSE au lieu de MAE) et un sampler différent (`RandomSampler` au lieu du TPE par défaut d'Optuna) par rapport à la recherche à 200 trials. La comparaison "200 vs 1000 trials" n'isole donc pas strictement l'effet du nombre d'essais — pour une comparaison rigoureuse, il faudrait garder la même fonction objectif (même métrique, même sampler) entre les deux runs. À garder en tête pour l'interprétation, ou à corriger si tu veux une comparaison plus propre.

In [ ]:
def objective_1000(trial):
    n_estimators = trial.suggest_int('n_estimators', 100, 1000)
    max_depth = trial.suggest_int('max_depth', 10, 50)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 32)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 32)
    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2'])
    criterion = trial.suggest_categorical('criterion', ["squared_error", "absolute_error", "friedman_mse", "poisson"])

    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        criterion=criterion,
        random_state=21
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return mean_squared_error(y_test, y_pred)

study_1000 = optuna.create_study(direction='minimize', sampler=optuna.samplers.RandomSampler(seed=42))
study_1000.optimize(objective_1000, n_trials=1000)

print("Best trial:")
trial = study_1000.best_trial
print("Value: {:.4f}".format(trial.value))
print("Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

**🐛 Bug corrigé ici** : dans la version originale du notebook, cette cellule réutilisait par erreur `best_n_estimators`/`best_max_depth`/... du run à **200 trials** au lieu des résultats du run à **1000 trials** (`study_1000.best_params`). Résultat : le "modèle à 1000 trials" était en réalité une copie du modèle à 200 trials, ce qui explique pourquoi ils semblaient identiques. Corrigé ci-dessous en utilisant bien `study_1000.best_params`, et en incluant `max_features`/`criterion` qui manquaient aussi dans le modèle final.

In [ ]:
# Construction du modèle final avec les VRAIS meilleurs hyperparamètres de l'étude à 1000 trials
best_params_1000 = study_1000.best_params

best_model_1000 = RandomForestRegressor(
    n_estimators=best_params_1000['n_estimators'],
    max_depth=best_params_1000['max_depth'],
    min_samples_split=best_params_1000['min_samples_split'],
    min_samples_leaf=best_params_1000['min_samples_leaf'],
    max_features=best_params_1000['max_features'],
    criterion=best_params_1000['criterion'],
    random_state=42
)
best_model_1000.fit(X_train, y_train)
y_pred_rfr_optuna_1000_trials = best_model_1000.predict(X_test)

results_optuna_1000 = modelresults(y_pred_rfr_optuna_1000_trials, 'Random Forest Optuna (1000 trials)')

In [ ]:
# Comparaison finale des 3 versions du modèle
results_df = pd.DataFrame([results_fit, results_optuna_200, results_optuna_1000])
print(results_df)

**À interpréter une fois relancé** : si les 1000 trials n'apportent pas d'amélioration notable par rapport aux 200 trials (une fois le bug corrigé), c'est un résultat valide à documenter — ça suggérerait que 200 trials suffisaient déjà à approcher l'optimum sur cet espace de recherche, pas besoin d'aller plus loin. Si au contraire ça améliore nettement, on garde `best_model_1000` comme modèle final pour la suite.

## 7. Diagnostic — Courbe d'apprentissage (learning curve)

La learning curve permet de vérifier si le modèle **sur-apprend** (overfitting — grand écart entre performance train et validation) ou **sous-apprend** (underfitting — les deux courbes restent mauvaises), et si donner plus de données aiderait.

**⚠️ Deux corrections apportées par rapport à la version initiale :**
1. `scoring="f1_macro"` était utilisé par erreur — c'est une métrique de **classification**, invalide ici puisque `co2_per_capita` est une cible continue (régression). Remplacé par `"neg_mean_absolute_error"`.
2. `learning_curve` utilise un `KFold` **non mélangé par défaut**. Or `X_train` est trié par pays puis par année — chaque fold de validation tombait donc sur des **pays entiers jamais vus**, ce qui teste en réalité la généralisation à un nouveau pays (question différente, et plus difficile) plutôt que la généralisation dans le temps qu'on veut mesurer ici. Corrigé avec `KFold(shuffle=True)`.

In [ ]:
from sklearn.model_selection import learning_curve, KFold

# On évalue ici le meilleur modèle trouvé par Optuna (200 trials, ou 1000 si il s'avère meilleur)
model_to_evaluate = best_model_200

cv = KFold(n_splits=5, shuffle=True, random_state=42)  # shuffle=True corrige le tri par pays/année

train_sizes, train_scores, val_scores = learning_curve(
    model_to_evaluate, X_train, y_train,
    train_sizes=np.linspace(0.1, 1.0, 8),
    cv=cv, scoring="neg_mean_absolute_error", n_jobs=-1, random_state=42
)

# neg_mean_absolute_error retourne des valeurs négatives (convention sklearn) -> on repasse en positif
train_mean, train_std = -train_scores.mean(axis=1), train_scores.std(axis=1)
val_mean, val_std = -val_scores.mean(axis=1), val_scores.std(axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_mean, "o-", label="Entraînement")
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15)
plt.plot(train_sizes, val_mean, "o-", label="Validation")
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.15)
plt.xlabel("Nombre d'exemples d'entraînement")
plt.ylabel("MAE (plus bas = mieux)")
plt.title("Courbe d'apprentissage — Random Forest optimisé")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("X_train :", len(X_train))
print("X_test  :", len(X_test))

**Grille de lecture** :
- Courbe d'entraînement plate proche de 0 = normal pour un Random Forest (grande capacité à mémoriser), ne pas s'inquiéter de ça en particulier.
- Ce qui compte : est-ce que l'écart train/validation se **réduit** quand le nombre d'exemples augmente (bon signe, plus de données aiderait) ou reste-t-il **constant** (il faudrait plutôt régulariser le modèle : réduire `max_depth`, augmenter `min_samples_leaf`) ?

## 8. Prochaines étapes

1. **Analyser les erreurs du modèle** : regarder par pays/année où le modèle se trompe le plus (on soupçonne les producteurs de pétrole comme l'Arabie Saoudite/Russie, avec des profils atypiques).
2. **Comparer avec/sans lags** : ré-entraîner un modèle sans les colonnes `_lag1/_lag2/_lag5/_ma5` pour quantifier concrètement l'apport de l'historique temporel.
3. **Split par pays (`GroupKFold`)** en complément du split temporel, pour tester la généralisation à des pays jamais vus du tout (question différente de la généralisation temporelle).
4. **(Optionnel) Réseau de neurones** : à tester une fois le pipeline Random Forest bien maîtrisé.
5. **(Optionnel) Variable politique** : enrichir le dataset avec le parti au pouvoir par pays/année (source possible : ParlGov) pour tester son influence sur `co2_per_capita`.
6. **Rédaction du rapport final** : EDA, méthodologie (choix de cible, gestion du leakage, split temporel justifié), résultats (comparaison modèles, feature importance, learning curve), limites (nature en partie synthétique du dataset, distinction émissions de production vs de consommation, difficulté sur les pays producteurs de pétrole).